# list to numpy 

In [ ]:
import numpy as np 

# declare 
data = np.array([1 , 2 , 5 , 5])
# append 
data = np.append(data , 10)
# extend
data = np.append(data , [5 , 6 , 7 ]) # same extend in list 
# sort 
data.sort() 
# reverse 
data = data[::-1]
# delete element have index 2 
data = np.delete(data , 2) # tại vị trí 2 
# delete first number 5 
delete_index = np.where(data == 5)[0][0]  # ==> [2 , 3]
delete_index2 = np.where(data == 5) # return a tuple 
print(delete_index2)
data = np.delete(data , delete_index) # ==> delete index 2 
# delete more 1 elm 
start_index = 1 
end_index = 3 
data = np.concatenate(data[:start_index] , data[end_index:])
# or 
data = np.delete(data , range(1 , 3))
# find first elm 
index = np.where(elm = 5)[0][0]
print(index)
# count frequence 
count = (data == 3).sum()
# return dimension 
dim = data.shape()
print(dim)
print(data)

# numpy array 
- homogeneous data 
- fixed data type 
- contigous memory 
- prefomance 

# IOU calculation (intersection over union)

In [2]:
boxA = [0 ,0 , 100 , 100]
boxB = [50 , 50 , 150 , 150]

def computeIoU(boxA, boxB):
	# determine the (x, y)-coordinates of the intersection rectangle
	xA = max(boxA[0], boxB[0])
	yA = max(boxA[1], boxB[1])
	xB = min(boxA[2], boxB[2])
	yB = min(boxA[3], boxB[3])

	interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1) # nếu 2 bouding box chồng lên nhau 
	# print("interArea", interArea)
    
	# compute the area of both the prediction and ground-truth
	# rectangles
	boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
	boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
	# compute the intersection over union by taking the intersection
	# area and dividing it by the sum of prediction + ground-truth
	# areas - the interesection area
	iou = interArea / float(boxAArea + boxBArea - interArea)
	# return the intersection over union value
	return iou

# IOU calculation on N - boxes : list and array 

In [3]:
# Example usage
import time 
boxes = []
for i in range(10000):
  boxes.append([i+1, i+1, i+1, i+1])

initialTime = time.time()

for i in range(1, len(boxes)):
  iou = computeIoU(boxes[0],boxes[i])
  # print("iou", iou)


# calculating execution time
print("Time taken by NumPy Arrays to perform multiplication:",
    (time.time() - initialTime),
    "seconds")

Time taken by NumPy Arrays to perform multiplication: 0.010608196258544922 seconds


In [ ]:
# Array solution
import time 
initialTime = time.time()
boxes = np.array(boxes)
scores = np.array(scores)

# Coordinates of bounding boxes
x1 = boxes[:, 0] # lấy tất cả phần tử ở cột 0 
y1 = boxes[:, 1] 
x2 = boxes[:, 2]      
y2 = boxes[:, 3]   
# x1 , y1 : coordinate top left  
# x2 , y2 : coordinate bottom right

# Compute the area of the bounding boxes
areas = (x2 - x1 + 1) * (y2 - y1 + 1) # tránh việc iou = 0 do toạ độ trùng nhau 

# Compute the intersection areas
xx1 = np.maximum(x1[0], x1[1:])
yy1 = np.maximum(y1[0], y1[1:])
xx2 = np.minimum(x2[0], x2[1:])
yy2 = np.minimum(y2[0], y2[1:])

# đảm bảo giá trị không âm tức bouding box không trùng nhau 
w = np.maximum(0, xx2 - xx1 + 1)
h = np.maximum(0, yy2 - yy1 + 1)

inter = w * h

# # Compute the IoU
iou = inter / (areas[0] + areas[1:] - inter)
# print(iou)

print("Time taken by NumPy Arrays to perform multiplication:",
    (time.time() - initialTime),
    "seconds")

# non - maxima suppresion : list and array 

In [ ]:
import numpy as np

def non_max_suppression(boxes, scores, threshold):
    """
    Perform non-maximum suppression.

    Parameters:
    boxes (numpy.ndarray): Array of bounding boxes, each defined by [x1, y1, x2, y2].
    scores (numpy.ndarray): Array of confidence scores for each bounding box.
    threshold (float): Overlap threshold for suppression.

    Returns:
    numpy.ndarray: Indices of bounding boxes to keep.
    """
    if len(boxes) == 0:
        return []

    # Coordinates of bounding boxes
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    # Compute area of bounding boxes
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)

    # Sort by scores
    order = scores.argsort()[::-1]
    # sort reverse because have -1 in the last 

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        # Compute IoU (Intersection over Union)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)

        intersection = w * h
        iou = intersection / (areas[i] + areas[order[1:]] - intersection)

        # Suppress bounding boxes with IoU over the threshold
        inds = np.where(iou <= threshold)[0] # return tuple chứa các mảng chỉ số cho mỗi dimen 
        order = order[inds + 1]

    return keep

# Example usage
boxes = np.array([
    [100, 100, 210, 210],
    [105, 105, 215, 215],
    [150, 150, 250, 250]
])

scores = np.array([0.9, 0.8, 0.7])
threshold = 0.5

keep_indices = non_max_suppression(boxes, scores, threshold)
kept_boxes = boxes[keep_indices]

print("Indices of boxes to keep:", keep_indices)
print("Boxes to keep:")
print(kept_boxes)


Indices of boxes to keep: [np.int64(0), np.int64(2)]
Boxes to keep:
[[100 100 210 210]
 [150 150 250 250]]
